# 02 — Data Preprocessing 

This notebook performs the complete preprocessing pipeline using the project's
predefined utility functions:

- `prepare_features_and_labels()` for feature extraction and label mapping  
- `BreastCancerPreprocessor` for MinMax scaling and optional L2 normalization  

This ensures consistency across the entire project, avoids duplicated code, and 
produces reproducible processed datasets ready for classical and quantum SVM models.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Add project root so that src/ works
sys.path.append(os.path.abspath(".."))

from src.data.preprocessing import (
    load_raw_data,
    prepare_features_and_labels,
    BreastCancerPreprocessor,
    get_train_test_split,
)

print("Imports OK")


In [ ]:
df = load_raw_data("../data/raw/data.csv")

X, y = prepare_features_and_labels(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("NaN in X?", np.isnan(X).any())
print("Columns after feature extraction:", X.shape[1])


In [ ]:
pre = BreastCancerPreprocessor(apply_l2=False)

X_processed = pre.fit_transform(X)

print("NaN in X_processed?", np.isnan(X_processed).any())
print("Min:", X_processed.min(), "Max:", X_processed.max())


In [ ]:
X_train, X_test, y_train, y_test = get_train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=True,
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test  shape:", X_test.shape, y_test.shape)


In [ ]:
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save csv
processed_df = pd.DataFrame(X_processed)
processed_df["diagnosis"] = y
processed_df.to_csv(os.path.join(OUTPUT_DIR, "processed_data.csv"), index=False)

# Save numpy arrays
np.save(os.path.join(OUTPUT_DIR, "X.npy"), X_processed)
np.save(os.path.join(OUTPUT_DIR, "y.npy"), y)

np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

print("Files saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))


In [ ]:
X_saved = np.load("../data/processed/X.npy")
X_train_saved = np.load("../data/processed/X_train.npy")

print("X_saved equals X_processed?  ", np.allclose(X_saved, X_processed))
print("X_train matches?             ", np.allclose(X_train_saved, X_train))

print("\nMax diff X:", np.nanmax(np.abs(X_saved - X_processed)))
print("Max diff Train:", np.nanmax(np.abs(X_train_saved - X_train)))


## Preprocessing Summary

- Successfully extracted 30 numeric features.
- Removed NaN-only columns automatically.
- Applied MinMax scaling (no L2).
- Achieved clean, NaN-free processed datasets.
- Saved:
  - processed_data.csv
  - X.npy, y.npy
  - X_train.npy, X_test.npy
  - y_train.npy, y_test.npy

This dataset is now ready for use in the classical SVM baseline.


In [ ]:
"""import os

folder = "../data/processed"

for fname in os.listdir(folder):
    path = os.path.join(folder, fname)
    if os.path.isfile(path):
        os.remove(path)

print("Processed folder cleaned.")"""


Processed folder cleaned.


In [ ]:
import os

folder = "../data/processed"
print(os.listdir(folder))


In [ ]:
import os
import numpy as np
import pandas as pd

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) Save processed data (X_processed, y, splits)
processed_df = pd.DataFrame(X_processed)
processed_df["diagnosis"] = y
processed_df.to_csv(os.path.join(OUTPUT_DIR, "processed_data.csv"), index=False)

np.save(os.path.join(OUTPUT_DIR, "X.npy"), X_processed)
np.save(os.path.join(OUTPUT_DIR, "y.npy"), y)

np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

print("Files saved in:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

# 2) Load back and compare IN THE SAME CELL
X_saved = np.load(os.path.join(OUTPUT_DIR, "X.npy"))
X_train_saved = np.load(os.path.join(OUTPUT_DIR, "X_train.npy"))

print("\nRoundtrip checks:")
print("X_saved equals X_processed?   ", np.allclose(X_saved, X_processed))
print("X_train_saved equals X_train? ", np.allclose(X_train_saved, X_train))

# Optional: show max absolute difference (should be 0.0)
print("Max |X_saved - X_processed|:", np.max(np.abs(X_saved - X_processed)))
print("Max |X_train_saved - X_train|:", np.max(np.abs(X_train_saved - X_train)))


In [ ]:
nan_counts = np.isnan(X).sum(axis=0)
print(nan_counts)
